# 07. Generalização e Diagnóstico de Modelos

**DCT1401 — Inteligência Artificial (UFRN 2026.2)**  
**Aluna:** Mariana dos Santos Batista Medeiros (Matrícula: 20230083988)  

**Atividade 02 — Avaliação Comparativa de Modelos**

Este notebook analisa a capacidade de generalização do Sistema Especialista, da Árvore de Decisão, da Random Forest e do Gradient Boosting, conforme a Seção 5.2 do enunciado.

A avaliação contempla as diferenças de desempenho entre treinamento e teste, validação cruzada de cinco folds, diagnóstico de overfitting e underfitting e discussão do compromisso entre viés e variância.

Na classificação, a métrica principal é o **F1-score**; na regressão, o **RMSE**.

São mantidos os mesmos seis atributos preditores, a divisão de 80% para treinamento e 20% para teste, a estratificação da classificação e a semente aleatória `random_state=42`.

Os modelos selecionados anteriormente serão reutilizados, sem novos ajustes de hiperparâmetros. A validação cruzada será realizada exclusivamente sobre o conjunto de treinamento, com pré-processamento ajustado dentro de cada fold.

**Configuração do ambiente — Google Colab**

Esta etapa conecta o Google Colab ao Google Drive e configura os caminhos necessários para acessar o dataset original e os módulos Python do projeto.

A execução pressupõe que a pasta `projeto-cardiovascular-ia-02` esteja localizada em `Meu Drive`, conforme as instruções disponíveis no `README.md` da Atividade 02.

O código verifica a existência do dataset e prepara o ambiente para a execução das etapas seguintes.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys
import importlib

# Conectar ao Google Drive
drive.mount("/content/drive")

# Diretório do projeto
PROJETO = Path(
    "/content/drive/MyDrive/projeto-cardiovascular-ia-02"
)

assert PROJETO.exists(), "Pasta do projeto não encontrada."

# Executar a partir da pasta de notebooks
os.chdir(PROJETO / "notebooks")

# Permitir importar os módulos do projeto
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))

# Conferir os arquivos necessários
arquivos_necessarios = [
    "data/raw/cardiovascular_disease_prediction.csv",
    "src/pipeline_data.py",
    "src/metrics.py",
    "src/expert_system.py",
    "rules/regras_classificacao.txt",
    "models/decision_tree_clf.joblib",
    "models/decision_tree_reg.joblib",
    "models/random_forest_clf.joblib",
    "models/random_forest_reg.joblib",
    "models/gradient_boosting_clf.joblib",
    "models/gradient_boosting_reg.joblib",
]

faltando = [
    arquivo
    for arquivo in arquivos_necessarios
    if not (PROJETO / arquivo).is_file()
]

assert not faltando, (
    f"Arquivos não encontrados: {faltando}"
)

print("Ambiente configurado com sucesso!")
print(f"Diretório atual: {Path.cwd()}")
print(f"Arquivos necessários conferidos: {len(arquivos_necessarios)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ambiente configurado com sucesso!
Diretório atual: /content/drive/MyDrive/projeto-cardiovascular-ia-02/notebooks
Arquivos necessários conferidos: 11


## 1. Recuperação dos dados e protocolo experimental

Esta etapa recupera o dataset original e reproduz a divisão dos dados utilizada nos notebooks anteriores, mantendo 80% dos registros para treinamento e 20% para teste, com estratificação na classificação e semente aleatória `random_state=42`.

São preservadas as seis variáveis preditoras selecionadas na Atividade 01, garantindo que a avaliação de generalização utilize os mesmos conjuntos empregados na construção e avaliação dos modelos.

O código também verifica as dimensões dos conjuntos e a ausência de sobreposição entre os índices de treinamento e teste, assegurando a consistência do protocolo experimental.

Não são realizados novos ajustes de hiperparâmetros nesta etapa.

In [ ]:
import numpy as np
import pandas as pd
import src.pipeline_data as pipeline_data

importlib.reload(pipeline_data)

(
    X_train,
    X_test,
    y_train_c,
    y_test_c,
    y_train_r,
    y_test_r,
    feat_cols
) = pipeline_data.carregar_e_dividir_dados_sem_leakage(
    "../data/raw/cardiovascular_disease_prediction.csv"
)

# Conferir o mesmo protocolo dos notebooks anteriores
assert X_train.shape == (16000, 6)
assert X_test.shape == (4000, 6)
assert len(feat_cols) == 6

assert X_train.index.intersection(
    X_test.index
).empty

assert X_train.index.equals(y_train_c.index)
assert X_test.index.equals(y_test_c.index)

assert X_train.index.equals(y_train_r.index)
assert X_test.index.equals(y_test_r.index)

print("DADOS RECUPERADOS COM SUCESSO")
print(f"Treinamento: {X_train.shape}")
print(f"Teste: {X_test.shape}")
print(f"Variáveis: {feat_cols}")

print("\nDistribuição da classe positiva:")
print(f"Treinamento: {np.mean(y_train_c):.2%}")
print(f"Teste: {np.mean(y_test_c):.2%}")

DADOS RECUPERADOS COM SUCESSO
Treinamento: (16000, 6)
Teste: (4000, 6)
Variáveis: ['Diabetes', 'Hypertension', 'Chest_Pain', 'ECG_Result', 'Blood_Glucose', 'Blood_Pressure_Systolic']

Distribuição da classe positiva:
Treinamento: 30.60%
Teste: 30.60%


## 2. Recuperação dos modelos validados

Nesta etapa, são recuperados os seis modelos de aprendizado de máquina previamente ajustados e salvos nos Notebooks 04 e 05.

O Sistema Especialista é reconstruído utilizando as 11 regras e o mecanismo de fallback definidos no Notebook 03. Suas frequências empíricas são calculadas exclusivamente sobre o treinamento.

A recuperação dos modelos garante que a análise de generalização utilize os mesmos classificadores e regressores avaliados no Notebook 06, sem realizar novos ajustes de hiperparâmetros.

In [ ]:
import joblib
import importlib
import src.expert_system as modulo_se

# Recarregar o módulo do Sistema Especialista
importlib.reload(modulo_se)

from src.expert_system import (
    BaseConhecimentoClassificacao,
    MotorInferenciaClassificacao
)

# -------- MODELOS DE CLASSIFICAÇÃO --------

modelos_classificacao = {
    "Árvore de Decisão": joblib.load(
        PROJETO / "models/decision_tree_clf.joblib"
    ),
    "Random Forest": joblib.load(
        PROJETO / "models/random_forest_clf.joblib"
    ),
    "Gradient Boosting": joblib.load(
        PROJETO / "models/gradient_boosting_clf.joblib"
    )
}

# -------- MODELOS DE REGRESSÃO --------

modelos_regressao = {
    "Árvore de Decisão": joblib.load(
        PROJETO / "models/decision_tree_reg.joblib"
    ),
    "Random Forest": joblib.load(
        PROJETO / "models/random_forest_reg.joblib"
    ),
    "Gradient Boosting": joblib.load(
        PROJETO / "models/gradient_boosting_reg.joblib"
    )
}

# -------- SISTEMA ESPECIALISTA --------

# Pré-processamento ajustado apenas no treinamento
X_train_se, X_test_se, preprocessador_se = (
    pipeline_data.preparar_dados_sistema_especialista(
        X_train,
        X_test
    )
)

# Recuperar a base de regras
base_se = BaseConhecimentoClassificacao(
    PROJETO / "rules/regras_classificacao.txt"
)

# Calcular as frequências somente no treinamento
base_se.analisar_no_treinamento(
    X_train_se,
    y_train_c
)

# Criar o motor de inferência
motor_se = MotorInferenciaClassificacao(base_se)

# -------- VALIDAÇÕES --------

assert len(modelos_classificacao) == 3
assert len(modelos_regressao) == 3
assert len(base_se.regras) == 11

assert X_train_se.shape == (16000, 6)
assert X_test_se.shape == (4000, 6)

print("MODELOS RECUPERADOS COM SUCESSO")

print("\nClassificação:")
for nome in modelos_classificacao:
    print(f" - {nome}")

print("\nRegressão:")
for nome in modelos_regressao:
    print(f" - {nome}")

print("\nSistema Especialista:")
print(f" - Regras carregadas: {len(base_se.regras)}")
print(f" - Fallback: classe {base_se.classe_padrao}")

print("\nTotal: 6 modelos de aprendizado de máquina")
print("       + 1 Sistema Especialista")

MODELOS RECUPERADOS COM SUCESSO

Classificação:
 - Árvore de Decisão
 - Random Forest
 - Gradient Boosting

Regressão:
 - Árvore de Decisão
 - Random Forest
 - Gradient Boosting

Sistema Especialista:
 - Regras carregadas: 11
 - Fallback: classe 0

Total: 6 modelos de aprendizado de máquina
       + 1 Sistema Especialista


## 3. Comparação entre treinamento e teste

A capacidade de generalização é analisada por meio da comparação das métricas obtidas nos conjuntos de treinamento e teste.

Na classificação, utiliza-se o F1-score; na regressão, o RMSE. O gap é calculado pela diferença entre o resultado no treinamento e o resultado no teste.

Para o F1-score, um gap positivo indica redução de desempenho no teste. Para o RMSE, um gap negativo indica aumento do erro no teste.

Os modelos são avaliados sem novos treinamentos ou ajustes. No Sistema Especialista, as frequências das regras foram estimadas exclusivamente a partir do conjunto de treinamento, preservando o procedimento adotado anteriormente.

O gap é um indicador de generalização, mas não permite, isoladamente, estabelecer um diagnóstico definitivo de overfitting ou underfitting. Essa análise será complementada pela validação cruzada e pela comparação com modelos de diferentes complexidades.

**Convenção adotada:** neste notebook e nas tabelas consolidadas do relatório final, todos os gaps são calculados como desempenho no treinamento menos desempenho na avaliação (teste ou validação). Para métricas de erro, como RMSE, valores negativos indicam aumento do erro na avaliação. Algumas análises dos Notebooks 04 e 05 apresentam a diferença inversa, validação menos treinamento; trata-se apenas de uma mudança na convenção do sinal, sem alteração dos resultados experimentais.

In [ ]:
from sklearn.metrics import f1_score, mean_squared_error
import numpy as np
import pandas as pd

pasta_tabelas = PROJETO / "reports" / "tables"
pasta_tabelas.mkdir(parents=True, exist_ok=True)

# -------- CLASSIFICAÇÃO --------

registros_gap_classificacao = []

# Sistema Especialista
pred_treino_se, _, _ = motor_se.inferir(X_train_se)
pred_teste_se, _, _ = motor_se.inferir(X_test_se)

f1_treino_se = f1_score(
    y_train_c, pred_treino_se, zero_division=0
)

f1_teste_se = f1_score(
    y_test_c, pred_teste_se, zero_division=0
)

registros_gap_classificacao.append({
    "Modelo": "Sistema Especialista",
    "F1 treino": f1_treino_se,
    "F1 teste": f1_teste_se,
    "Gap (treino - teste)": f1_treino_se - f1_teste_se
})

# Árvore de Decisão e ensembles
for nome, modelo in modelos_classificacao.items():

    pred_treino = modelo.predict(X_train)
    pred_teste = modelo.predict(X_test)

    f1_treino = f1_score(
        y_train_c, pred_treino, zero_division=0
    )

    f1_teste = f1_score(
        y_test_c, pred_teste, zero_division=0
    )

    registros_gap_classificacao.append({
        "Modelo": nome,
        "F1 treino": f1_treino,
        "F1 teste": f1_teste,
        "Gap (treino - teste)": f1_treino - f1_teste
    })

tabela_gap_classificacao = pd.DataFrame(
    registros_gap_classificacao
)

assert len(tabela_gap_classificacao) == 4

tabela_gap_classificacao.to_csv(
    pasta_tabelas / "generalizacao_classificacao.csv",
    index=False,
    float_format="%.16g"
)

print("GENERALIZAÇÃO — CLASSIFICAÇÃO (F1-SCORE)")
display(tabela_gap_classificacao.round(4))


# -------- REGRESSÃO --------

registros_gap_regressao = []

for nome, modelo in modelos_regressao.items():

    pred_treino = modelo.predict(X_train)
    pred_teste = modelo.predict(X_test)

    rmse_treino = np.sqrt(
        mean_squared_error(y_train_r, pred_treino)
    )

    rmse_teste = np.sqrt(
        mean_squared_error(y_test_r, pred_teste)
    )

    registros_gap_regressao.append({
        "Modelo": nome,
        "RMSE treino": rmse_treino,
        "RMSE teste": rmse_teste,
        "Gap (treino - teste)": rmse_treino - rmse_teste
    })

tabela_gap_regressao = pd.DataFrame(
    registros_gap_regressao
)

assert len(tabela_gap_regressao) == 3

tabela_gap_regressao.to_csv(
    pasta_tabelas / "generalizacao_regressao.csv",
    index=False,
    float_format="%.16g"
)

print("\nGENERALIZAÇÃO — REGRESSÃO (RMSE)")
display(tabela_gap_regressao.round(4))

print("\nTabelas de generalização salvas com sucesso!")

GENERALIZAÇÃO — CLASSIFICAÇÃO (F1-SCORE)


,Modelo,F1 treino,F1 teste,Gap (treino - teste)
0,Sistema Especialista,0.6414,0.6224,0.0191
1,Árvore de Decisão,0.6350,0.6172,0.0178
2,Random Forest,0.6316,0.6064,0.0252
3,Gradient Boosting,0.6285,0.6129,0.0157



GENERALIZAÇÃO — REGRESSÃO (RMSE)


,Modelo,RMSE treino,RMSE teste,Gap (treino - teste)
0,Árvore de Decisão,2.9681,2.9707,-0.0027
1,Random Forest,2.9328,2.9529,-0.0201
2,Gradient Boosting,2.9502,2.9526,-0.0024



Tabelas de generalização salvas com sucesso!


### 3.1 Interpretação dos gaps de generalização

Na classificação, todos os modelos apresentaram F1-score menor no teste do que no treinamento. Os gaps variaram de 0,0157 a 0,0252, indicando reduções relativamente pequenas de desempenho.

A Random Forest apresentou o maior gap (0,0252), enquanto o Gradient Boosting apresentou o menor (0,0157). O Sistema Especialista apresentou gap de 0,0191, sugerindo comportamento relativamente estável entre os dois conjuntos.

Na regressão, os gaps foram negativos porque o RMSE foi ligeiramente maior no teste. A Random Forest apresentou a maior diferença (-0,0201), enquanto a Árvore de Decisão (-0,0027) e o Gradient Boosting (-0,0024) apresentaram diferenças menores.

Esses resultados não evidenciam grandes perdas de desempenho nos dados de teste. Entretanto, gaps pequenos não comprovam, isoladamente, ausência de overfitting ou underfitting. O diagnóstico será complementado pela validação cruzada de cinco folds e pela análise do compromisso entre viés e variância.

## 4. Validação cruzada de cinco folds

A validação cruzada permite avaliar a estabilidade dos modelos em diferentes divisões do conjunto de treinamento.

Na classificação, utiliza-se StratifiedKFold com cinco folds, preservando aproximadamente a proporção das classes. Na regressão, utiliza-se KFold com cinco folds.

Em cada rodada, o Pipeline completo é ajustado novamente sobre quatro folds e avaliado no fold restante. Esse procedimento mantém o pré-processamento restrito aos dados de treinamento de cada rodada, evitando vazamento de informações.

A métrica principal é o F1-score para classificação e o RMSE para regressão. São apresentados a média e o desvio padrão dos resultados de validação, além das médias de treinamento.

Os hiperparâmetros permanecem fixos, conforme selecionados anteriormente. Como foram escolhidos utilizando validação cruzada sobre o mesmo conjunto de treinamento, os resultados desta análise devem ser interpretados como uma avaliação complementar de estabilidade, e não como uma estimativa independente e livre de viés de seleção.

In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)
import numpy as np
import pandas as pd

cv_classificacao = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

registros_cv_classificacao = []

for nome, modelo in modelos_classificacao.items():

    print(f"Avaliando: {nome}")

    resultado_cv = cross_validate(
        estimator=modelo,
        X=X_train,
        y=y_train_c,
        cv=cv_classificacao,
        scoring="f1",
        return_train_score=True,
        n_jobs=2,
        error_score="raise"
    )

    f1_treino = resultado_cv["train_score"]
    f1_validacao = resultado_cv["test_score"]

    registros_cv_classificacao.append({
        "Modelo": nome,
        "F1 médio treino CV": f1_treino.mean(),
        "F1 médio validação CV": f1_validacao.mean(),
        "Desvio padrão CV": f1_validacao.std(ddof=0),
        "Gap treino - validação CV": (
            f1_treino.mean() - f1_validacao.mean()
        )
    })

tabela_cv_classificacao = pd.DataFrame(
    registros_cv_classificacao
)

assert len(tabela_cv_classificacao) == 3

tabela_cv_classificacao.to_csv(
    pasta_tabelas / "generalizacao_cv_classificacao.csv",
    index=False,
    float_format="%.16g"
)

print("\nVALIDAÇÃO CRUZADA — CLASSIFICAÇÃO (5 FOLDS)")
display(tabela_cv_classificacao.round(4))

print("\nTabela de validação cruzada salva com sucesso!")

Avaliando: Árvore de Decisão
Avaliando: Random Forest
Avaliando: Gradient Boosting

VALIDAÇÃO CRUZADA — CLASSIFICAÇÃO (5 FOLDS)


,Modelo,F1 médio treino CV,F1 médio validação CV,Desvio padrão CV,Gap treino - validação CV
0,Árvore de Decisão,0.6350,0.6349,0.0057,0.0000
1,Random Forest,0.6312,0.6248,0.0061,0.0064
2,Gradient Boosting,0.6286,0.6239,0.0033,0.0048



Tabela de validação cruzada salva com sucesso!


### 4.1 Interpretação da validação cruzada — Classificação

Na validação cruzada de cinco folds, a Árvore de Decisão apresentou o maior F1-score médio de validação (0,6349), seguida pela Random Forest (0,6248) e pelo Gradient Boosting (0,6239).

Os gaps entre treinamento e validação foram pequenos. A Árvore de Decisão apresentou diferença praticamente nula, enquanto Random Forest e Gradient Boosting apresentaram gaps de 0,0064 e 0,0048, respectivamente.

O Gradient Boosting apresentou o menor desvio padrão (0,0033), indicando maior estabilidade entre os folds analisados. A Random Forest apresentou o maior desvio padrão (0,0061), embora a variação também tenha sido pequena.

Os resultados não evidenciam sobreajuste acentuado durante a validação cruzada. Entretanto, gaps pequenos não descartam underfitting, e a comparação com o teste final continua necessária.

Como os hiperparâmetros foram selecionados anteriormente por validação cruzada no mesmo conjunto de treinamento, esses resultados são interpretados como análise complementar de estabilidade, e não como uma estimativa totalmente independente de generalização.

In [ ]:
from sklearn.model_selection import KFold, cross_validate
import numpy as np
import pandas as pd

# Validação cruzada com cinco folds
cv_regressao = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

registros_cv_regressao = []

for nome, modelo in modelos_regressao.items():

    print(f"Avaliando: {nome}")

    resultado_cv = cross_validate(
        estimator=modelo,
        X=X_train,
        y=y_train_r,
        cv=cv_regressao,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
        n_jobs=2,
        error_score="raise"
    )

    # Converter os erros negativos para RMSE positivo
    rmse_treino = -resultado_cv["train_score"]
    rmse_validacao = -resultado_cv["test_score"]

    assert len(rmse_treino) == 5
    assert len(rmse_validacao) == 5

    assert np.isfinite(rmse_treino).all()
    assert np.isfinite(rmse_validacao).all()

    registros_cv_regressao.append({
        "Modelo": nome,
        "RMSE médio treino CV": rmse_treino.mean(),
        "RMSE médio validação CV": rmse_validacao.mean(),
        "Desvio padrão CV": rmse_validacao.std(ddof=0),
        "Gap treino - validação CV": (
            rmse_treino.mean() - rmse_validacao.mean()
        )
    })

# Construir a tabela comparativa
tabela_cv_regressao = pd.DataFrame(
    registros_cv_regressao
)

assert len(tabela_cv_regressao) == 3

# Salvar os resultados
tabela_cv_regressao.to_csv(
    pasta_tabelas / "generalizacao_cv_regressao.csv",
    index=False,
    float_format="%.16g"
)

print("\nVALIDAÇÃO CRUZADA — REGRESSÃO (5 FOLDS)")
display(tabela_cv_regressao.round(4))

print("\nTabela de validação cruzada salva com sucesso!")

Avaliando: Árvore de Decisão
Avaliando: Random Forest
Avaliando: Gradient Boosting

VALIDAÇÃO CRUZADA — REGRESSÃO (5 FOLDS)


,Modelo,RMSE médio treino CV,RMSE médio validação CV,Desvio padrão CV,Gap treino - validação CV
0,Árvore de Decisão,2.9641,2.9834,0.0398,-0.0194
1,Random Forest,2.9282,2.9703,0.0408,-0.0421
2,Gradient Boosting,2.9479,2.9627,0.0406,-0.0147



Tabela de validação cruzada salva com sucesso!


### 4.2 Interpretação da validação cruzada — Regressão

Na validação cruzada de cinco folds, o Gradient Boosting apresentou o menor RMSE médio de validação (2,9627), seguido pela Random Forest (2,9703) e pela Árvore de Decisão (2,9834).

Os três modelos apresentaram RMSE maior na validação do que no treinamento, resultando em gaps negativos. A Random Forest apresentou o maior gap em magnitude (-0,0421), enquanto o Gradient Boosting apresentou o menor (-0,0147).

Os desvios padrão dos modelos ficaram próximos de 0,04, indicando variação relativamente pequena do RMSE entre os cinco folds analisados.

Os resultados sugerem generalização relativamente estável, sem evidência de sobreajuste acentuado. Entretanto, a proximidade entre treinamento e validação não descarta underfitting, que também depende da capacidade preditiva alcançada e da complexidade do modelo.

Como os hiperparâmetros foram selecionados anteriormente no mesmo conjunto de treinamento, esses resultados são interpretados como análise complementar de estabilidade, não como uma estimativa independente de generalização.

### 4.3 Validação cruzada do Sistema Especialista

O Sistema Especialista também é submetido à validação cruzada estratificada de cinco folds, permitindo comparar sua estabilidade com a dos classificadores de aprendizado de máquina.

As 11 regras e sua ordem de prioridade permanecem fixas. Em cada fold, o pré-processamento é ajustado exclusivamente sobre a parcela de treinamento, e as frequências empíricas das regras e o mecanismo de fallback são recalculados utilizando apenas os rótulos dessa parcela.

Em seguida, o sistema realiza inferências sobre os dados de treinamento e validação do fold. O F1-score é calculado para ambos os conjuntos.

Esse procedimento evita vazamento de informações entre treinamento e validação e permite obter a média, o desvio padrão e o gap do Sistema Especialista, sem utilizar o conjunto de teste final.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

cv_especialista = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

registros_cv_especialista = []

for numero_fold, (idx_treino, idx_validacao) in enumerate(
    cv_especialista.split(X_train, y_train_c),
    start=1
):

    # Separar os dados brutos deste fold
    X_treino_fold = X_train.iloc[idx_treino]
    X_validacao_fold = X_train.iloc[idx_validacao]

    y_treino_fold = y_train_c.iloc[idx_treino]
    y_validacao_fold = y_train_c.iloc[idx_validacao]

    # Pré-processamento ajustado somente no treino do fold
    X_treino_se_fold, X_validacao_se_fold, _ = (
        pipeline_data.preparar_dados_sistema_especialista(
            X_treino_fold,
            X_validacao_fold
        )
    )

    # Criar uma base de conhecimento independente para o fold
    base_fold = BaseConhecimentoClassificacao(
        PROJETO / "rules/regras_classificacao.txt"
    )

    # Recalcular frequências e fallback somente no treino
    base_fold.analisar_no_treinamento(
        X_treino_se_fold,
        y_treino_fold
    )

    motor_fold = MotorInferenciaClassificacao(base_fold)

    # Inferência nos dois conjuntos
    pred_treino_fold, _, _ = motor_fold.inferir(
        X_treino_se_fold
    )

    pred_validacao_fold, _, _ = motor_fold.inferir(
        X_validacao_se_fold
    )

    # Calcular F1-score
    f1_treino_fold = f1_score(
        y_treino_fold,
        pred_treino_fold,
        zero_division=0
    )

    f1_validacao_fold = f1_score(
        y_validacao_fold,
        pred_validacao_fold,
        zero_division=0
    )

    registros_cv_especialista.append({
        "Fold": numero_fold,
        "F1 treino": f1_treino_fold,
        "F1 validação": f1_validacao_fold,
        "Gap": f1_treino_fold - f1_validacao_fold
    })

    print(
        f"Fold {numero_fold}: "
        f"F1 treino={f1_treino_fold:.4f} | "
        f"F1 validação={f1_validacao_fold:.4f}"
    )

# Tabela com os cinco folds
tabela_folds_especialista = pd.DataFrame(
    registros_cv_especialista
)

assert len(tabela_folds_especialista) == 5

# Resumo comparável aos modelos de ML
tabela_cv_especialista = pd.DataFrame([{
    "Modelo": "Sistema Especialista",
    "F1 médio treino CV": (
        tabela_folds_especialista["F1 treino"].mean()
    ),
    "F1 médio validação CV": (
        tabela_folds_especialista["F1 validação"].mean()
    ),
    "Desvio padrão CV": (
        tabela_folds_especialista["F1 validação"].std(ddof=0)
    ),
    "Gap treino - validação CV": (
        tabela_folds_especialista["Gap"].mean()
    )
}])

# Salvar resultados
tabela_folds_especialista.to_csv(
    pasta_tabelas / "generalizacao_cv_especialista_folds.csv",
    index=False,
    float_format="%.16g"
)

tabela_cv_especialista.to_csv(
    pasta_tabelas / "generalizacao_cv_especialista.csv",
    index=False,
    float_format="%.16g"
)

print("\nVALIDAÇÃO CRUZADA — SISTEMA ESPECIALISTA")
display(tabela_cv_especialista.round(4))

print("\nValidação cruzada concluída e salva!")

Fold 1: F1 treino=0.6407 | F1 validação=0.6442
Fold 2: F1 treino=0.6433 | F1 validação=0.6339
Fold 3: F1 treino=0.6408 | F1 validação=0.6437
Fold 4: F1 treino=0.6402 | F1 validação=0.6463
Fold 5: F1 treino=0.6421 | F1 validação=0.6387

VALIDAÇÃO CRUZADA — SISTEMA ESPECIALISTA


,Modelo,F1 médio treino CV,F1 médio validação CV,Desvio padrão CV,Gap treino - validação CV
0,Sistema Especialista,0.6414,0.6414,0.0045,0.0



Validação cruzada concluída e salva!


### 4.4 Interpretação da validação cruzada — Sistema Especialista

O Sistema Especialista apresentou F1-score médio de validação de 0,6414, com desvio padrão de 0,0045. As médias de treinamento e validação foram praticamente iguais, resultando em gap próximo de zero.

Entretanto, o F1-score do teste final (0,6224) ficou abaixo da média de validação cruzada. Essa redução não ocorreu apenas no Sistema Especialista: também foi observada na Árvore de Decisão, na Random Forest e no Gradient Boosting, com diferenças aproximadas de 0,018, 0,018 e 0,011, respectivamente. O comportamento pode estar associado à variabilidade entre as partições e ao fato de o teste final apresentar maior dificuldade para os classificadores avaliados.

Há uma limitação metodológica relevante: as regras do Sistema Especialista foram elaboradas anteriormente utilizando o conjunto completo de treinamento. Embora as frequências empíricas tenham sido recalculadas dentro de cada fold, a estrutura das regras permaneceu influenciada pelos dados posteriormente utilizados na validação cruzada.

Consequentemente, essa validação serve para analisar a estabilidade das regras fixas, mas não é uma estimativa totalmente independente de seu processo de elaboração. O conjunto de teste final oferece uma referência mais adequada para avaliar o sistema em registros não utilizados na construção das regras.

## 5. Comparação consolidada da validação cruzada

Nesta seção, são reunidos os resultados da validação cruzada dos modelos de classificação e regressão.

Para a classificação, comparam-se os quatro modelos utilizando o F1-score médio, seu desvio padrão e o gap entre treinamento e validação.

Para a regressão, comparam-se os três modelos utilizando o RMSE médio, seu desvio padrão e o gap correspondente.

A consolidação permite identificar diferenças de desempenho e estabilidade, oferecendo suporte à análise posterior de overfitting, underfitting e compromisso entre viés e variância.

In [ ]:
import pandas as pd
import numpy as np

# -------- CLASSIFICAÇÃO --------

comparacao_cv_classificacao = pd.concat(
    [
        tabela_cv_especialista,
        tabela_cv_classificacao
    ],
    ignore_index=True
)

assert len(comparacao_cv_classificacao) == 4
assert comparacao_cv_classificacao["Modelo"].is_unique

# -------- REGRESSÃO --------

comparacao_cv_regressao = tabela_cv_regressao.copy()

assert len(comparacao_cv_regressao) == 3
assert comparacao_cv_regressao["Modelo"].is_unique

# -------- VERIFICAÇÕES --------

for tabela in [
    comparacao_cv_classificacao,
    comparacao_cv_regressao
]:
    valores = tabela.select_dtypes(include="number")
    assert np.isfinite(valores.to_numpy()).all()

# -------- SALVAR RESULTADOS --------

comparacao_cv_classificacao.to_csv(
    pasta_tabelas / "generalizacao_cv_classificacao_final.csv",
    index=False,
    float_format="%.16g"
)

comparacao_cv_regressao.to_csv(
    pasta_tabelas / "generalizacao_cv_regressao_final.csv",
    index=False,
    float_format="%.16g"
)

# -------- EXIBIR TABELAS --------

print("COMPARAÇÃO FINAL — CLASSIFICAÇÃO")
display(comparacao_cv_classificacao.round(4))

print("\nCOMPARAÇÃO FINAL — REGRESSÃO")
display(comparacao_cv_regressao.round(4))

print("\nTabelas consolidadas salvas com sucesso!")

COMPARAÇÃO FINAL — CLASSIFICAÇÃO


,Modelo,F1 médio treino CV,F1 médio validação CV,Desvio padrão CV,Gap treino - validação CV
0,Sistema Especialista,0.6414,0.6414,0.0045,0.0000
1,Árvore de Decisão,0.6350,0.6349,0.0057,0.0000
2,Random Forest,0.6312,0.6248,0.0061,0.0064
3,Gradient Boosting,0.6286,0.6239,0.0033,0.0048



COMPARAÇÃO FINAL — REGRESSÃO


,Modelo,RMSE médio treino CV,RMSE médio validação CV,Desvio padrão CV,Gap treino - validação CV
0,Árvore de Decisão,2.9641,2.9834,0.0398,-0.0194
1,Random Forest,2.9282,2.9703,0.0408,-0.0421
2,Gradient Boosting,2.9479,2.9627,0.0406,-0.0147



Tabelas consolidadas salvas com sucesso!


### 5.1 Síntese da validação cruzada

Na classificação, o Sistema Especialista apresentou o maior F1-score médio de validação (0,6414), seguido pela Árvore de Decisão (0,6349), Random Forest (0,6248) e Gradient Boosting (0,6239).

Esse ranking deve ser interpretado com cautela, pois as regras do Sistema Especialista foram elaboradas utilizando o conjunto completo de treinamento, que também forneceu os folds de validação. Portanto, sua avaliação cruzada não é metodologicamente equivalente a uma validação independente do processo de elaboração das regras.

Os gaps entre treinamento e validação foram pequenos, variando de aproximadamente zero a 0,0064. O Gradient Boosting apresentou o menor desvio padrão (0,0033), enquanto a Random Forest apresentou o maior (0,0061).

Na regressão, o Gradient Boosting alcançou o menor RMSE médio de validação (2,9627), seguido pela Random Forest (2,9703) e pela Árvore de Decisão (2,9834). Os gaps foram negativos, indicando erros ligeiramente maiores na validação, com desvios padrão próximos de 0,04.

Em conjunto, os resultados sugerem estabilidade e ausência de sobreajuste acentuado nos modelos finais. Entretanto, diferenças pequenas entre treinamento e validação não descartam automaticamente underfitting.

Os resultados de validação cruzada também devem ser considerados complementares, pois os hiperparâmetros dos modelos de aprendizado de máquina já haviam sido selecionados utilizando o mesmo conjunto de treinamento. As diferenças observadas não constituem, isoladamente, evidência de superioridade estatística.

## 6. Diagnóstico de overfitting, underfitting e ajuste adequado

O diagnóstico de generalização considera conjuntamente o desempenho no treinamento, no teste e na validação cruzada.

Como referência para investigar o efeito da complexidade, são recuperados os resultados das árvores de decisão irrestritas avaliadas no Notebook 04. Esses modelos são comparados às respectivas árvores ajustadas, mantendo as mesmas variáveis e a divisão original dos dados.

A comparação permite observar se a maior complexidade melhora o desempenho no treinamento à custa de pior generalização, fornecendo evidências quantitativas para a discussão sobre overfitting e o compromisso entre viés e variância.

O diagnóstico dos demais modelos será fundamentado nas métricas de generalização e validação cruzada, evitando classificações automáticas baseadas apenas em limites arbitrários de gap.

In [ ]:
# Recuperar os resultados das árvores irrestritas
arvores_irrestritas = pd.read_csv(
    pasta_tabelas / "arvores_comparacao_irrestritas.csv",
    encoding="utf-8-sig"
)

assert len(arvores_irrestritas) == 2

# Comparar a classificação
arvore_classificacao = tabela_gap_classificacao.loc[
    tabela_gap_classificacao["Modelo"] == "Árvore de Decisão"
].iloc[0]

arvore_irrestrita_c = arvores_irrestritas.loc[
    arvores_irrestritas["Modelo"] == "Classificação irrestrita"
].iloc[0]

comparacao_arvores_c = pd.DataFrame([
    {
        "Modelo": "Árvore irrestrita",
        "F1 treino": arvore_irrestrita_c["Treinamento"],
        "F1 teste": arvore_irrestrita_c["Teste"]
    },
    {
        "Modelo": "Árvore ajustada",
        "F1 treino": arvore_classificacao["F1 treino"],
        "F1 teste": arvore_classificacao["F1 teste"]
    }
])

comparacao_arvores_c["Gap treino - teste"] = (
    comparacao_arvores_c["F1 treino"]
    - comparacao_arvores_c["F1 teste"]
)

# Comparar a regressão
arvore_regressao = tabela_gap_regressao.loc[
    tabela_gap_regressao["Modelo"] == "Árvore de Decisão"
].iloc[0]

arvore_irrestrita_r = arvores_irrestritas.loc[
    arvores_irrestritas["Modelo"] == "Regressão irrestrita"
].iloc[0]

comparacao_arvores_r = pd.DataFrame([
    {
        "Modelo": "Árvore irrestrita",
        "RMSE treino": arvore_irrestrita_r["Treinamento"],
        "RMSE teste": arvore_irrestrita_r["Teste"]
    },
    {
        "Modelo": "Árvore ajustada",
        "RMSE treino": arvore_regressao["RMSE treino"],
        "RMSE teste": arvore_regressao["RMSE teste"]
    }
])

# Padronizar a definição do gap: treino - teste
comparacao_arvores_r["Gap treino - teste"] = (
    comparacao_arvores_r["RMSE treino"]
    - comparacao_arvores_r["RMSE teste"]
)

# Salvar resultados consolidados
comparacao_arvores_c.to_csv(
    pasta_tabelas / "generalizacao_arvores_classificacao.csv",
    index=False,
    float_format="%.16g"
)

comparacao_arvores_r.to_csv(
    pasta_tabelas / "generalizacao_arvores_regressao.csv",
    index=False,
    float_format="%.16g"
)

print("ÁRVORES — CLASSIFICAÇÃO")
display(comparacao_arvores_c.round(4))

print("\nÁRVORES — REGRESSÃO")
display(comparacao_arvores_r.round(4))

print("\nComparações salvas com sucesso!")

ÁRVORES — CLASSIFICAÇÃO


,Modelo,F1 treino,F1 teste,Gap treino - teste
0,Árvore irrestrita,0.8876,0.5218,0.3658
1,Árvore ajustada,0.6350,0.6172,0.0178



ÁRVORES — REGRESSÃO


,Modelo,RMSE treino,RMSE teste,Gap treino - teste
0,Árvore irrestrita,1.6213,4.0296,-2.4083
1,Árvore ajustada,2.9681,2.9707,-0.0027



Comparações salvas com sucesso!


### 6.1 Comparação entre árvores irrestritas e ajustadas

A comparação evidencia sobreajuste nas árvores de decisão irrestritas.

Na classificação, a árvore irrestrita apresentou F1-score de 0,8876 no treinamento e 0,5218 no teste, produzindo gap de 0,3658. Após o ajuste de hiperparâmetros, o gap diminuiu para 0,0178, e o F1-score de teste aumentou para 0,6172.

Na regressão, a árvore irrestrita apresentou RMSE de 1,6213 no treinamento e 4,0296 no teste, com gap de -2,4083. A árvore ajustada reduziu significativamente essa diferença, alcançando RMSE de 2,9681 no treinamento e 2,9707 no teste.

Os resultados demonstram que as árvores irrestritas apresentaram melhor ajuste aos dados de treinamento, mas pior desempenho em dados não vistos, caracterizando overfitting.

A redução de complexidade aumentou o erro de treinamento, mas melhorou a generalização. Esse comportamento ilustra o compromisso entre viés e variância: modelos excessivamente flexíveis podem capturar particularidades do treinamento, enquanto a regularização tende a reduzir essa sensibilidade.

Embora as árvores ajustadas apresentem gaps pequenos, isso não comprova ausência de underfitting. O diagnóstico também depende da capacidade preditiva alcançada, da complexidade do modelo e das limitações das variáveis disponíveis.

## 7. Diagnóstico de overfitting, underfitting e compromisso entre viés e variância

O diagnóstico considera conjuntamente os gaps entre treinamento e teste, a validação cruzada de cinco folds, a capacidade preditiva observada e os experimentos de complexidade realizados nos Notebooks 04 e 05.

Para atender à classificação solicitada no enunciado, os modelos finais são categorizados como **bem ajustados**, considerando as condições experimentais avaliadas. Essa classificação indica ausência de evidência relevante de sobreajuste e não representa garantia de desempenho em qualquer conjunto de dados.

### 7.1 Diagnóstico individual dos modelos

| Modelo | Diagnóstico | Evidência numérica |
|---|---|---|
| Sistema Especialista — classificação | **Bem ajustado** | Gap F1 treino–teste de 0,0191; gap CV próximo de zero |
| Árvore de Decisão — classificação | **Bem ajustada** | Gap F1 de 0,0178; profundidades maiores não melhoraram a validação cruzada |
| Random Forest — classificação | **Bem ajustada** | Gap F1 de 0,0252; gap CV de 0,0064 |
| Gradient Boosting — classificação | **Bem ajustado** | Gap F1 de 0,0157; gap CV de 0,0048 |
| Árvore de Decisão — regressão | **Bem ajustada** | Gap RMSE de -0,0027; RMSE de teste de 2,9707 |
| Random Forest — regressão | **Bem ajustada** | Gap RMSE de -0,0201; RMSE médio CV de 2,9703 |
| Gradient Boosting — regressão | **Bem ajustado** | Gap RMSE de -0,0024; RMSE médio CV de 2,9627 |

Como referência, as árvores irrestritas apresentaram **overfitting**: a árvore de classificação teve gap F1 de 0,3658, enquanto a árvore de regressão apresentou RMSE de 1,6213 no treinamento e 4,0296 no teste.

Os modelos finais apresentaram gaps pequenos. Além disso, os experimentos com árvores irrestritas e as buscas de hiperparâmetros mostram que aumentar a complexidade não produziu os melhores resultados de validação nas configurações avaliadas.

Assim, não foram identificadas evidências suficientes para classificar os modelos finais como underfitting corrigível simplesmente por maior complexidade. Entretanto, isso não exclui outras formas de subajuste ou limitações de representação.

O desempenho preditivo moderado pode estar relacionado às informações disponíveis nas seis variáveis, à variabilidade dos dados e às características dos algoritmos. Essas possibilidades não foram isoladas experimentalmente e, portanto, não podem ser estabelecidas como causas comprovadas.

### 7.2 Compromisso entre viés e variância por família

**Sistema Especialista:** apresenta regras fixas e interpretáveis, o que limita a flexibilidade para representar relações complexas e pode introduzir viés estrutural. Como as decisões seguem regras predefinidas, sua estrutura tende a variar menos do que a de modelos aprendidos diretamente dos dados. O desvio padrão de F1 de 0,0045 na validação cruzada indica estabilidade do desempenho entre folds, mas não mede diretamente a variância das previsões. O diagnóstico experimental é de modelo bem ajustado, com limitações estruturais de flexibilidade.

**Árvore de Decisão:** a versão irrestrita apresentou forte sobreajuste, comportamento compatível com alta variância. A limitação de profundidade, o tamanho mínimo das folhas e a poda reduziram a complexidade, aumentando o erro de treinamento, mas melhorando o desempenho no teste. Esse resultado exemplifica a troca de parte da flexibilidade do modelo por maior capacidade de generalização, ilustrando o compromisso entre viés e variância.

**Random Forest:** combina árvores treinadas com amostras bootstrap e agrega suas previsões, mecanismo que busca reduzir a variância em relação a uma árvore isolada. A curva de erro do Notebook 05 mostrou melhoria inicial e posterior estabilização com o aumento do número de árvores, comportamento compatível com os benefícios e limites da agregação. Na configuração selecionada, `max_features=1.0`, todas as variáveis podem participar de cada divisão; portanto, não há subamostragem aleatória de atributos por nó, e o funcionamento se aproxima do Bagging. A profundidade máxima de 6 e o mínimo de 10 amostras por folha também restringem a complexidade das árvores-base.

**Gradient Boosting:** constrói árvores sequencialmente para corrigir erros das etapas anteriores, buscando reduzir o viés do conjunto de previsões. Como esse processo também pode aumentar a sensibilidade a particularidades do treinamento, a profundidade das árvores, a taxa de aprendizado, a quantidade de estimadores e o `subsample` são importantes controles de complexidade. Nos experimentos realizados, os modelos selecionados apresentaram gaps pequenos, sem evidência de sobreajuste acentuado.

### 7.3 Limites do diagnóstico

A classificação como bem ajustado é fundamentada nas métricas e nos experimentos realizados, não em uma decomposição matemática direta de viés e variância.

A seleção dos hiperparâmetros e a avaliação cruzada utilizaram o mesmo conjunto de treinamento. Além disso, as regras do Sistema Especialista foram elaboradas antes da validação cruzada, com acesso aos dados posteriormente utilizados nos folds.

Consequentemente, a comparação considera essas limitações metodológicas e preserva o conjunto de teste final como referência para o desempenho em registros não utilizados durante a construção dos modelos.

## 8. Síntese da análise de generalização

A análise de generalização contemplou os sete modelos finais, utilizando gaps entre treinamento e teste, validação cruzada de cinco folds e comparação de modelos com diferentes níveis de complexidade.

**Na classificação**, os gaps de F1-score entre treinamento e teste variaram de 0,0157 a 0,0252. O Sistema Especialista apresentou o maior F1 médio de validação cruzada (0,6414), seguido pela Árvore de Decisão (0,6349), Random Forest (0,6248) e Gradient Boosting (0,6239). Entretanto, esse ranking deve ser interpretado com cautela, pois a estrutura das regras do Sistema Especialista foi elaborada utilizando o conjunto completo de treinamento, inclusive os registros posteriormente utilizados nos folds de validação.

No teste final, o Sistema Especialista também obteve o maior F1-score (0,6224), embora a diferença em relação aos demais classificadores tenha sido pequena.

**Na regressão**, o Gradient Boosting apresentou o menor RMSE médio de validação (2,9627), seguido pela Random Forest (2,9703) e pela Árvore de Decisão (2,9834). Os três modelos apresentaram pequenas diferenças entre os resultados de treinamento e teste.

A comparação entre árvores irrestritas e ajustadas demonstrou claramente o efeito do overfitting. Na classificação, o gap de F1-score caiu de 0,3658 para 0,0178 após o ajuste. Na regressão, o gap de RMSE passou de -2,4083 para -0,0027, acompanhado de melhoria expressiva no desempenho de teste.

**Os sete modelos finais foram classificados como bem ajustados nas condições experimentais avaliadas**, pois apresentaram gaps pequenos e não exibiram a deterioração expressiva observada nas árvores irrestritas. Não foram identificadas evidências suficientes para classificá-los como underfitting decorrente apenas de complexidade insuficiente.

O compromisso entre viés e variância foi observado principalmente nas árvores de decisão. No Sistema Especialista, destacou-se a limitação estrutural das regras fixas; na Random Forest, a agregação de árvores busca reduzir a variância; e no Gradient Boosting, o aprendizado sequencial busca reduzir o viés, com mecanismos de regularização para controlar o risco de sobreajuste.

Essas interpretações são qualitativas. Não foi realizada decomposição matemática direta de viés e variância, e o desempenho moderado pode estar associado a diferentes fatores, incluindo a informação disponível nas seis variáveis preditoras.

A análise utiliza uma única partição final de teste, e a validação cruzada dos modelos de aprendizado de máquina foi realizada após a seleção de hiperparâmetros sobre o mesmo conjunto de treinamento. Essas limitações devem ser consideradas na interpretação dos resultados.

A próxima etapa, apresentada no Notebook 08, avaliará a eficiência computacional dos modelos, incluindo tempo de treinamento, inferência, memória, tamanho em disco, complexidade e estimativas de energia e emissão de CO₂.